# Detección de enfermedades en hojas de maíz — versión Google Colab (GPU)
**Curso:** Inteligencia Artificial – USAT | **TA3: Machine Learning**

Compara tres modelos: **CNN propia**, **MobileNetV2** y **ResNet50**.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar*.

Ejecuta las celdas en orden (Shift+Enter). Tiempo total aproximado: 25–40 min.

## 0. Verificar la GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('PyTorch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No hay GPU: activa GPU T4 en Entorno de ejecución > Cambiar tipo'

## 1. Conectar Google Drive
Los modelos y resultados se guardan en `MyDrive/maiz_ia_salida`, así no se pierden si se desconecta la sesión.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['MAIZ_SALIDA'] = '/content/drive/MyDrive/maiz_ia_salida'
os.makedirs(os.environ['MAIZ_SALIDA'], exist_ok=True)
print('Salida en:', os.environ['MAIZ_SALIDA'])

## 2. Cargar el código del proyecto
Sube `proyecto_maiz_ia.zip` a la raíz de tu Google Drive (o súbelo cuando la celda lo pida).

In [ ]:
import os, zipfile
ZIP_DRIVE = '/content/drive/MyDrive/proyecto_maiz_ia.zip'
if os.path.exists(ZIP_DRIVE):
    ruta_zip = ZIP_DRIVE
else:
    from google.colab import files
    print('No se encontró el zip en Drive. Selecciona proyecto_maiz_ia.zip:')
    subido = files.upload()
    ruta_zip = list(subido.keys())[0]
zipfile.ZipFile(ruta_zip).extractall('/content')
%cd /content/maiz_ia
!ls

## 3. Descargar el dataset PlantVillage desde Kaggle
Se descarga directo a Colab (≈2 GB, 1–3 min). Si `kagglehub` pide credenciales, ver la celda alternativa.

In [ ]:
!pip -q install kagglehub
import kagglehub
RUTA_DATASET = kagglehub.dataset_download('abdallahalidev/plantvillage-dataset')
print('Dataset en:', RUTA_DATASET)

**Alternativa (solo si la celda anterior falla):** en Kaggle → tu perfil → *Settings → API → Create New Token* descarga `kaggle.json`. Quita el `#` de las líneas y ejecuta.

In [ ]:
# from google.colab import files; files.upload()   # sube kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/pv --unzip
# RUTA_DATASET = '/content/pv'

## 4. Preparar el dataset (filtrar maíz, deduplicar, split 70/15/15)
Primero se copian solo las 4 carpetas de maíz (≈3 852 imágenes) a `/content/maiz_raw`; así no se recorren las ~160 000 imágenes del dataset completo. Tarda menos de 1 minuto.

In [ ]:
import glob, os, shutil
carpetas = [d for d in glob.glob(f'{RUTA_DATASET}/*/color/Corn*') + glob.glob(f'{RUTA_DATASET}/color/Corn*')
            if os.path.isdir(d)]
print('Carpetas de maíz encontradas:', len(carpetas))
for d in carpetas: print('  ', os.path.basename(d))
assert len(carpetas) == 4, 'No se encontraron las 4 carpetas: revisa RUTA_DATASET con !ls'
shutil.rmtree('/content/maiz_raw', ignore_errors=True)
for d in carpetas:
    shutil.copytree(d, f'/content/maiz_raw/{os.path.basename(d)}')
!python 1_preparar_dataset.py --origen /content/maiz_raw

In [ ]:
from IPython.display import Image, display
SAL = os.environ['MAIZ_SALIDA'] + '/resultados'
display(Image(f'{SAL}/distribucion_dataset.png'))
display(Image(f'{SAL}/muestras_dataset.png'))

## 5. Entrenamiento de los tres modelos
Puedes entrenarlos por separado (recomendado, para ver cada uno) o todos juntos con `--modelo todos`.

Tiempos aproximados en T4: CNN 4–8 min · MobileNetV2 5–8 min · ResNet50 10–15 min.

### 5.1 CNN propia (línea base, desde cero)

In [ ]:
!python 2_entrenar.py --modelo cnn

In [ ]:
display(Image(f'{SAL}/cnn_curvas.png'))

### 5.2 MobileNetV2 (transfer learning, ligero)

In [ ]:
!python 2_entrenar.py --modelo mobilenet

In [ ]:
display(Image(f'{SAL}/mobilenet_curvas.png'))

### 5.3 ResNet50 (transfer learning, profundo)

In [ ]:
!python 2_entrenar.py --modelo resnet

In [ ]:
display(Image(f'{SAL}/resnet_curvas.png'))

## 6. Evaluación en el conjunto de prueba y comparación

In [ ]:
!python 3_evaluar.py

In [ ]:
import pandas as pd
tabla = pd.read_csv(f'{SAL}/comparacion_modelos.csv')
display(tabla)
display(Image(f'{SAL}/comparacion_modelos.png'))

In [ ]:
for m in ['cnn', 'mobilenet', 'resnet']:
    print(open(f'{SAL}/{m}_reporte.txt', encoding='utf-8').read())
    display(Image(f'{SAL}/{m}_matriz_confusion.png'))

In [ ]:
# Ejemplos mal clasificados (para la discusión)
import glob
for ruta in sorted(glob.glob(f'{SAL}/*_errores.png')):
    print(ruta); display(Image(ruta))

## 7. Probar con tus propias fotos
Sube una o varias fotos de hojas de maíz (por ejemplo, tomadas en campo). La app de escritorio `4_app.py` hace lo mismo en tu laptop.

In [ ]:
import torch, matplotlib.pyplot as plt
from PIL import Image as PILImage
from torchvision import transforms
from google.colab import files
import config
from modelos import cargar_checkpoint

MODELO = 'mobilenet'   # cambia a 'resnet' o 'cnn'
modelo, ckpt = cargar_checkpoint(f"{os.environ['MAIZ_SALIDA']}/modelos/{MODELO}_best.pt", 'cpu')
tf = transforms.Compose([transforms.Resize((ckpt['img_size'],)*2), transforms.ToTensor(),
                         transforms.Normalize(config.MEAN, config.STD)])
for nombre in files.upload():
    img = PILImage.open(nombre).convert('RGB')
    with torch.no_grad():
        p = torch.softmax(modelo(tf(img).unsqueeze(0)), 1)[0]
    i = int(p.argmax())
    plt.figure(figsize=(4, 4)); plt.imshow(img); plt.axis('off')
    plt.title(f"{config.CLASES_ES[ckpt['clases'][i]]}\n{p[i]:.1%}", fontsize=10); plt.show()
    print({c: f'{v:.1%}' for c, v in zip(ckpt['clases'], p.tolist())})

## 8. Descargar resultados
Todo ya está en `MyDrive/maiz_ia_salida`. Esta celda además genera un zip para descargar.

Para usar la app en tu laptop: copia la carpeta `modelos/` dentro de tu carpeta `maiz_ia/` y ejecuta `python 4_app.py --modelo mobilenet` (o `resnet`).

In [ ]:
import shutil
shutil.make_archive('/content/resultados_maiz', 'zip', os.environ['MAIZ_SALIDA'])
files.download('/content/resultados_maiz.zip')